# Visual-language assistant with Gemma3 and OpenVINO
![](https://github.com/user-attachments/assets/2540a58e-c242-4439-b151-0fd1e6938af1)

Gemma 3 is Google's new iteration of open weight LLMs. It comes in four sizes, 1 billion, 4 billion, 12 billion, and 27 billion parameters with base (pre-trained) and instruction-tuned versions. The 4, 12, and 27 billion parameter models can process both images and text, while the 1B variant is text only.

The input context window length has been increased from Gemma 2’s 8k to 32k for the 1B variants, and 128k for all others. As is the case with other VLMs (vision-language models), Gemma 3 generates text in response to the user inputs, which may consist of text and, optionally, images. Example uses include question answering, analyzing image content, summarizing documents, etc.

The three core enhancements in Gemma 3 over Gemma 2 are:
* Longer context length
* Multimodality
* Multilinguality

You can find more details about model in the [blog post](https://developers.googleblog.com/en/introducing-gemma3/).

In this tutorial we consider how to convert and optimize Gemma3 model for creating multimodal chatbot using [Optimum Intel](https://github.com/huggingface/optimum-intel). Additionally, we demonstrate how to apply model optimization techniques like weights compression using [NNCF](https://github.com/openvinotoolkit/nncf).

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Select model](#Select-model)
- [Convert and Optimize model](#Convert-and-Optimize-model)
    - [Compress model weights to 4-bit](#Compress-model-weights-to-4-bit)
    - [Select inference device](#Select-inference-device)
- [Run model inference](#Run-model-inference)
- [Interactive demo](#Interactive-demo)


### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/gemma3/gemma3.ipynb" />


## Prerequisites
[back to top ⬆️](#Table-of-contents:)

In [1]:
import platform

%pip install -q "torch>=2.1" "torchvision" "Pillow" "gradio>=4.36" "opencv-python" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install  -q -U "openvino>=2025.0.0" "openvino-tokenizers>=2025.0.0" "nncf>=2.15.0"
%pip install -q "git+https://github.com/huggingface/optimum-intel.git" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -q "git+https://github.com/huggingface/transformers@v4.55.4-Gemma-3" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install openvino-genai
%pip install optimum-intel@git+https://github.com/huggingface/optimum-intel.git
%pip install nncf

if platform.system() == "Darwin":
    %pip install -q "numpy<2.0"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
toto-ts 0.1.3 requires torch==2.7.0, but you have torch 2.8.0 which is incompatible.
toto-ts 0.1.3 requires transformers==4.52.1, but you have transformers 4.50.0.dev0 which is incompatible.
gluonts 0.15.1 requires numpy~=1.16, but you have numpy 2.2.6 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
  error: subprocess-exit

In [2]:
from pathlib import Path
import requests

if not Path("cmd_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py")
    open("cmd_helper.py", "w").write(r.text)

if not Path("notebook_utils.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py")
    open("notebook_utils.py", "w").write(r.text)

## Select model
[back to top ⬆️](#Table-of-contents:)

There are multiple Gemma3 models available in [models collection](https://huggingface.co/collections/google/gemma-3-release-67c6c6f89c4f76621268bb6d).
>**Note**: run model with notebook, you will need to accept license agreement. 
>You must be a registered user in 🤗 Hugging Face Hub. Please visit selected model card, e.g. [google/gemma-3-4b-it](https://huggingface.co/google/gemma-3-4b-it), carefully read terms of usage and click accept button.  You will need to use an access token for the code below to run. For more information on access tokens, refer to [this section of the documentation](https://huggingface.co/docs/hub/security-tokens).
>You can login on Hugging Face Hub in notebook environment, using following code:

In [3]:
# uncomment these lines to login to huggingfacehub to get access to pretrained model

#from huggingface_hub import notebook_login, whoami

#try:
#   whoami()
#   print('Authorization token already provided')
#except OSError:
#   notebook_login() 

In [4]:
import ipywidgets as widgets

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("gemma3.ipynb")

model_ids = ["google/gemma-3-4b-it", "google/gemma-3-12b-it"]

model_id = widgets.Dropdown(
    options=model_ids,
    default=model_ids[0],
    description="Model:",
)

model_id

Dropdown(description='Model:', options=('google/gemma-3-4b-it', 'google/gemma-3-12b-it'), value='google/gemma-…

In [5]:
print(f"Selected {model_id.value}")
pt_model_id = model_id.value
model_dir = Path(pt_model_id.split("/")[-1])

Selected google/gemma-3-4b-it


## Convert and Optimize model
[back to top ⬆️](#Table-of-contents:)

Gemma3 is PyTorch model. OpenVINO supports PyTorch models via conversion to OpenVINO Intermediate Representation (IR). [OpenVINO model conversion API](https://docs.openvino.ai/2025/openvino-workflow/model-preparation.html#convert-a-model-with-python-convert-model) should be used for these purposes. `ov.convert_model` function accepts original PyTorch model instance and example input for tracing and returns `ov.Model` representing this model in OpenVINO framework. Converted model can be used for saving on disk using `ov.save_model` function or directly loading on device using `core.compile_model`. 

For convenience, we will use OpenVINO integration with HuggingFace Optimum. 🤗 [Optimum Intel](https://huggingface.co/docs/optimum/intel/index) is the interface between the 🤗 Transformers and Diffusers libraries and the different tools and libraries provided by Intel to accelerate end-to-end pipelines on Intel architectures.

Among other use cases, Optimum Intel provides a simple interface to optimize your Transformers and Diffusers models, convert them to the OpenVINO Intermediate Representation (IR) format and run inference using OpenVINO Runtime. `optimum-cli` provides command line interface for model conversion and optimization. 

General command format:

```bash
optimum-cli export openvino --model <model_id_or_path> --task <task> <output_dir>
```

where task is task to export the model for, if not specified, the task will be auto-inferred based on the model. You can find a mapping between tasks and model classes in Optimum TaskManager [documentation](https://huggingface.co/docs/optimum/exporters/task_manager). Additionally, you can specify weights compression using `--weight-format` argument with one of following options: `fp32`, `fp16`, `int8` and `int4`. Fro int8 and int4 [nncf](https://github.com/openvinotoolkit/nncf) will be used for  weight compression. More details about model export provided in [Optimum Intel documentation](https://huggingface.co/docs/optimum/intel/openvino/export#export-your-model).


### Compress model weights to 4-bit
[back to top ⬆️](#Table-of-contents:)
For reducing memory consumption, weights compression optimization can be applied using [NNCF](https://github.com/openvinotoolkit/nncf). 

<details>
    <summary><b>Click here for more details about weight compression</b></summary>
Weight compression aims to reduce the memory footprint of a model. It can also lead to significant performance improvement for large memory-bound models, such as Large Language Models (LLMs). LLMs and other models, which require extensive memory to store the weights during inference, can benefit from weight compression in the following ways:

* enabling the inference of exceptionally large models that cannot be accommodated in the memory of the device;

* improving the inference performance of the models by reducing the latency of the memory access when computing the operations with weights, for example, Linear layers.

[Neural Network Compression Framework (NNCF)](https://github.com/openvinotoolkit/nncf) provides 4-bit / 8-bit mixed weight quantization as a compression method primarily designed to optimize LLMs. The main difference between weights compression and full model quantization (post-training quantization) is that activations remain floating-point in the case of weights compression which leads to a better accuracy. Weight compression for LLMs provides a solid inference performance improvement which is on par with the performance of the full model quantization. In addition, weight compression is data-free and does not require a calibration dataset, making it easy to use.

`nncf.compress_weights` function can be used for performing weights compression. The function accepts an OpenVINO model and other compression parameters. Compared to INT8 compression, INT4 compression improves performance even more, but introduces a minor drop in prediction quality.

More details about weights compression, can be found in [OpenVINO documentation](https://docs.openvino.ai/2025/openvino-workflow/model-optimization-guide/weight-compression.html).
</details>

In [6]:
to_compress = widgets.Checkbox(value=True, description="Compress model")
to_compress

Checkbox(value=True, description='Compress model')

In [7]:
from cmd_helper import optimum_cli

model_export_dir = model_dir / ("INT4" if to_compress.value else "FP16")

if not model_export_dir.exists():
    optimum_cli(pt_model_id, model_export_dir, additional_args={"weight-format": "int4" if to_compress.value else "fp16"})

**Export command:**

`optimum-cli export openvino --model google/gemma-3-4b-it gemma-3-4b-it/INT4 --weight-format int4`

b'/Users/emlanza/Repos/GitHub repos/openvino_notebooks/openvino_env/lib/python3.12/site-packages/torch/onnx/_internal/registration.py:162: OnnxExporterWarning: Symbolic function \'aten::scaled_dot_product_attention\' already registered for opset 14. Replacing the existing function with new function. This is unexpected. Please report it on https://github.com/pytorch/pytorch/issues.\n  warnings.warn(\nTraceback (most recent call last):\n  File "/Users/emlanza/Repos/GitHub repos/openvino_notebooks/openvino_env/lib/python3.12/site-packages/huggingface_hub/utils/_http.py", line 409, in hf_raise_for_status\n    response.raise_for_status()\n  File "/Users/emlanza/Repos/GitHub repos/openvino_notebooks/openvino_env/lib/python3.12/site-packages/requests/models.py", line 1026, in raise_for_status\n    raise HTTPError(http_error_msg, response=self)\nrequests.exceptions.HTTPError: 401 Client Error: Unauthorized for url: https://huggingface.co/api/models/google/gemma-3-4b-it/tree/main?recursive=True

CalledProcessError: Command '['optimum-cli', 'export', 'openvino', '--model', 'google/gemma-3-4b-it', 'gemma-3-4b-it/INT4', '--weight-format', 'int4']' returned non-zero exit status 1.

### Select inference device
[back to top ⬆️](#Table-of-contents:)

To run the inference, **OpenVINO GenAI** is a library from Intel designed to efficiently run large language models (LLMs) and multimodal models (models that handle both text and images) on Intel hardware. It provides a high-level API that simplifies inference by automatically handling tasks like tokenization, preprocessing, and decoding. Models can be loaded from local files or directly from Hugging Face or optimized OpenVINO format for better performance. The library also supports device selection (CPU, GPU, or other Intel accelerators). This makes it easier for developers to integrate advanced AI models into applications while taking advantage of Intel hardware optimizations. Go [HERE](https://openvinotoolkit.github.io/openvino.genai/) to know more.


![image](ov_genai.png)

### Select the device to run the inference

In [ ]:

from notebook_utils import device_widget

device = device_widget(default="AUTO", exclude=["NPU"])

device

## Run model inference
[back to top ⬆️](#Table-of-contents:)

In [ ]:
import openvino_genai as ov_genai
import openvino as ov
from PIL import Image
import numpy as np
from pathlib import Path
from io import BytesIO

In [ ]:
# Load the image

def load_image(path_or_url):
    import os

    if isinstance(path_or_url, str) and path_or_url.startswith(("http", "https")):
        import requests
        response = requests.get(path_or_url)
        if response.status_code != 200 or not response.content:
            raise FileNotFoundError(f"Could not retrieve image from URL: {path_or_url}")
        try:
            return Image.open(BytesIO(response.content)).convert("RGB")
        except Exception as e:
            raise FileNotFoundError(f"Failed to open image from URL: {path_or_url}") from e
    else:
        # Accept Path or str for local files
        file_path = str(path_or_url)
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"Image file does not exist: {file_path}")
        try:
            return Image.open(file_path).convert("RGB")
        except Exception as e:
            raise FileNotFoundError(f"Failed to open image file: {file_path}") from e

image_url = "https://github.com/openvinotoolkit/openvino_notebooks/assets/29454499/d5fbbd1a-d484-415c-88cb-9986625b7b11"
image_file = Path("cat.png")

image = load_image(image_file)

# Convert image to tensor
images_tensor = [ov.Tensor(np.array(image)[None])]  # Add batch dimension



In [ ]:
# Create a openvino genai pipeline for the model

pipe = ov_genai.VLMPipeline(model_export_dir, device=device.value) 

In [ ]:
# Run the inference

prompt = "Describe what is unusual in these images."
display(image)
result = pipe.generate(prompt, images=images_tensor, max_new_tokens=100)
print(result.texts[0])

## Interactive demo
[back to top ⬆️](#Table-of-contents:)

In [8]:
if not Path("gradio_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/notebooks/gemma3/gradio_helper.py")
    open("gradio_helper.py", "w").write(r.text)

In [9]:
from gradio_helper import make_demo

demo = make_demo(pipe)

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(share=True, debug=True)
# if you are launching remotely, specify server_name and server_port
# demo.launch(server_name='your server name', server_port='server port in int')
# Read more in the docs: https://gradio.app/docs/

NameError: name 'pipe' is not defined